# Evidence optimization of source regularization

This tutorial implements the Bayesian procedure of Suyu et al. (2006). Instead of choosing the regularization strength by eye, we optimize $\eta=\log\lambda$ using the source-marginalized evidence. We also compare proper zeroth, gradient, and curvature priors using their evidence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from akira import (
    CartesianGrid, EllipticalPowerLaw, ExternalShear, LensPlane, SourceGrid,
    forward_operator, gaussian_psf, lensing_operator, optimize_regularization,
    simulate_imaging,
)

## A higher-resolution source problem

The source now has $20\times20=400$ pixels rather than the original 100. The physical source-plane extent is unchanged. This is a substantial increase in flexibility, but still a more controlled demonstration than 1,600 source pixels fitted to 900 image pixels.

In [ ]:
image_grid = CartesianGrid((30, 30), pixel_scale=0.08)
source_grid = SourceGrid((20, 20), pixel_scale=0.03, centre=(0.03, -0.02))
lens = LensPlane((
    EllipticalPowerLaw(0.72, slope=2.08, q=0.78, phi=0.25),
    ExternalShear(0.025, phi=0.65),
))
coordinates = source_grid.coordinates()
x, y = coordinates[..., 0], coordinates[..., 1]
true_source = (
    np.exp(-0.5*((x-0.06)**2/0.07**2 + (y+0.03)**2/0.045**2))
    + 0.35*np.exp(-0.5*((x+0.08)**2/0.035**2 + (y-0.06)**2/0.05**2))
)
unblurred_operator = lensing_operator(image_grid, source_grid, lens)
unblurred = (unblurred_operator @ true_source.reshape(-1)).reshape(image_grid.shape)
observation = simulate_imaging(
    unblurred, gaussian_psf(0.7, size=5), noise_sigma=0.008,
    pixel_scale=image_grid.pixel_scale, seed=7,
)
operator = forward_operator(image_grid, source_grid, lens, observation.psf.kernel)
print(f'{observation.data.size} data pixels; {source_grid.size} source pixels')

## Proper Suyu regularization matrices

The gradient prior includes zero-valued exterior conditions on the top and right edges. Curvature uses second differences internally, gradient terms at those edges, and zero-value terms on the final boundary pixels. These are equations A2 and A3 of Suyu et al.; the boundary terms remove constant and planar null modes, making the Gaussian prior normalized and its determinant meaningful.

## Optimize $\log\lambda$

For every trial $\eta=\log\lambda$, Akira solves the source and evaluates

$$\log P(d|\lambda,F,H)=-E_D-\lambda E_S-\frac12\log\det A+\frac12\log\det(\lambda H)-\frac12\log\det C_D-\frac{N_d}{2}\log(2\pi).$$

Optimizing the logarithm is numerically natural because plausible strengths span orders of magnitude.

In [ ]:
results = {
    kind: optimize_regularization(
        observation, source_grid, operator, kind,
        initial_log_strength=np.log(30.0),
        log_strength_bounds=(-8.0, 20.0),
    )
    for kind in ('zeroth', 'gradient', 'curvature')
}
print('kind          lambda    log evidence   effective N_s   stationarity')
for kind, result in results.items():
    print(f'{kind:10s} {result.strength:10.3f} {result.log_evidence:15.2f} '          f'{result.effective_source_parameters:15.2f} '          f'{result.stationarity_residual:13.2e}')

The effective source dimension is

$$\gamma=N_s-\lambda\,\mathrm{Tr}(A^{-1}H).$$

It counts source combinations meaningfully constrained by the data. At the optimum, Suyu equation (20) requires $\lambda s^THs=\gamma$; the small stationarity residual verifies this independently. Comparing optimized evidence values then selects the regularization form, including its Occam penalty.

In [ ]:
source_extent = (-0.27, 0.33, -0.32, 0.28)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
panels = [('truth', true_source)] + [
    (kind, np.asarray(result.reconstruction.source))
    for kind, result in results.items()
]
vmin = min(values.min() for _, values in panels)
vmax = max(values.max() for _, values in panels)
for axis, (title, values) in zip(axes, panels):
    shown = axis.imshow(values, origin='lower', extent=source_extent, cmap='magma',
                        vmin=vmin, vmax=vmax)
    axis.set_title(title); axis.set_xlabel('source x [arcsec]')
axes[0].set_ylabel('source y [arcsec]')
fig.colorbar(shown, ax=axes, fraction=0.02); plt.show()

In [ ]:
print('kind        source RMSE   chi-squared')
for kind, result in results.items():
    reconstructed = np.asarray(result.reconstruction.source)
    rmse = np.sqrt(np.mean((reconstructed - true_source)**2))
    print(f'{kind:10s} {rmse:11.4f} {result.reconstruction.chi_squared:13.2f}')
preferred_kind = max(results, key=lambda kind: results[kind].log_evidence)
print(f'\nEvidence-preferred regularization: {preferred_kind}')

This evidence optimization can also be activated during free-lens fitting with `fit_pixelized_model(..., objective='evidence', optimize_regularization_strength=True)`. That jointly optimizes the lens parameters and $\log\lambda$. For very fine grids the current bilinear operator is only piecewise differentiable at source-pixel boundaries, so convergence diagnostics remain essential; sparse/matrix-free operators and more robust nonlinear exploration are later production milestones.